# Day 023 Project: Article Scraper

## What You're Building

A `BookScraper` class that fetches [books.toscrape.com](https://books.toscrape.com) — a site built specifically for scraping practice — extracts structured book data using CSS selectors, and uses the LLM to provide insights about the catalog.

The same class structure applies to any content site: swap the CSS selectors and you have a scraper for Hacker News, a news aggregator, or a product catalogue.

## Project Requirements

1. Implement `BookScraper` with a `requests.Session` and a User-Agent header
2. `get_soup(url=None)` — fetch a page and return BeautifulSoup
3. `extract_books(soup)` — extract `{title, price, rating}` from each article
4. `ai_insights(books, question)` — LLM analysis of the extracted catalog

**Deliverable:** Run `scraper.extract_books(soup)`, print 5 titles, and get an AI answer to a question about the catalog.

In [ ]:
import requests
import json
import ollama
from bs4 import BeautifulSoup

## Provided: All Helper Functions

In [ ]:
def parse_html(html_string: str) -> BeautifulSoup:
    return BeautifulSoup(html_string, "html.parser")


def find_all_links(soup: BeautifulSoup) -> list[dict]:
    links = []
    for a in soup.find_all("a", href=True):
        links.append({"text": a.get_text(strip=True), "href": a["href"]})
    return links


def extract_by_selector(soup: BeautifulSoup, css_selector: str) -> list[str]:
    return [
        el.get_text(strip=True)
        for el in soup.select(css_selector)
        if el.get_text(strip=True)
    ]


def fetch_and_parse(url: str, headers: dict | None = None) -> BeautifulSoup:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser")


def ai_extract_from_page(html_content: str, question: str, model: str = "llama3.2") -> str:
    soup = BeautifulSoup(html_content, "html.parser")
    for tag in soup(["script", "style"]):
        tag.decompose()
    text = soup.get_text(separator="\n", strip=True)
    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "system",
                "content": "You are a web page analyst. Answer questions about the page content concisely.",
            },
            {
                "role": "user",
                "content": f"Page content:\n{text[:3000]}\n\nQuestion: {question}",
            },
        ],
    )
    return response["message"]["content"]

## Your Implementation

Implement `BookScraper` using the helper functions above and CSS selectors.

**Useful selectors on books.toscrape.com:**
- `article.product_pod` — each book card
- `h3 a` — link with `title` attribute (full book title)
- `p.price_color` — price text
- `p.star-rating` — rating via `class` list (e.g. `['star-rating', 'Three']`)

In [ ]:
SCRAPE_URL = "https://books.toscrape.com"


class BookScraper:
    BASE = SCRAPE_URL

    def __init__(self):
        # TODO: self.session = requests.Session()
        # TODO: self.session.headers['User-Agent'] = 'Mozilla/5.0 (educational scraper)'
        pass

    def get_soup(self, url: str | None = None) -> BeautifulSoup:
        # TODO: url = url or self.BASE
        # TODO: r = self.session.get(url, timeout=10)
        # TODO: r.raise_for_status()
        # TODO: return BeautifulSoup(r.text, 'html.parser')
        pass

    def extract_books(self, soup: BeautifulSoup) -> list[dict]:
        # TODO: loop over soup.select('article.product_pod')
        #       for each: select_one('h3 a'), select_one('p.price_color'),
        #                 select_one('p.star-rating')
        #       return list of {title, price, rating} dicts
        pass

    def ai_insights(self, books: list[dict], question: str,
                    model: str = 'llama3.2') -> str:
        # TODO: json.dumps(books[:10]) → pass to ollama.chat as context
        # TODO: return response['message']['content']
        pass

## Use Your Scraper

In [ ]:
# 1. Create the scraper
# scraper = BookScraper()

# 2. Fetch the front page
# soup = scraper.get_soup()

# 3. Extract books
# books = scraper.extract_books(soup)
# print(f'Found {len(books)} books')
# for b in books[:5]:
#     print(f"  {b['title'][:50]} | {b['price']} | {b['rating']} stars")


In [ ]:
# 4. AI insights
# answer = scraper.ai_insights(books, 'What price ranges do you see in this catalog?')
# print('\nAI Insights:')
# print(answer)


## Checks

In [ ]:
def _run_project_checks():
    total = 5
    passed = 0

    # Check 1: BookScraper class with required methods
    try:
        assert 'BookScraper' in globals(), 'BookScraper not defined'
        for m in ('get_soup', 'extract_books', 'ai_insights'):
            assert hasattr(BookScraper, m), f'BookScraper missing method: {m}'
        passed += 1; print('\u2705 Check 1: BookScraper has all required methods')
    except Exception as e:
        print(f'\u274c Check 1: {e}')

    # Check 2: scraper is a BookScraper instance
    try:
        assert 'scraper' in globals(), 'scraper not defined'
        assert isinstance(scraper, BookScraper), \
            f'scraper must be BookScraper, got {type(scraper)}'
        passed += 1; print('\u2705 Check 2: scraper is a BookScraper')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: soup is a BeautifulSoup
    try:
        assert 'soup' in globals(), 'soup not defined'
        assert isinstance(soup, BeautifulSoup), \
            f'soup must be BeautifulSoup, got {type(soup)}'
        passed += 1; print('\u2705 Check 3: soup is a BeautifulSoup')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: books is a non-empty list with correct keys
    try:
        assert 'books' in globals(), 'books not defined'
        assert isinstance(books, list) and len(books) > 0, \
            f'books must be non-empty list, got {books!r}'
        for key in ('title', 'price', 'rating'):
            assert key in books[0], f"books[0] missing key '{key}': {books[0]}"
        passed += 1; print(f'\u2705 Check 4: books has {len(books)} items with title/price/rating')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: answer is a non-empty string
    try:
        assert 'answer' in globals(), 'answer not defined'
        assert isinstance(answer, str) and len(answer) > 10, \
            f'answer must be non-empty string, got {answer!r}'
        passed += 1; print(f'\u2705 Check 5: answer is {len(answer)} chars')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Project complete!')
    print(f'\nScore: {passed}/{total}')


_run_project_checks()

## Bonus Challenges

- Add a `scrape_all_pages(max_pages=5)` method that follows the 'next' link on each page to collect books from multiple pages
- Add a `save_csv(books, path)` method that writes the catalog to CSV (using the csv module from Day 21)
- Try extracting from a different books.toscrape.com category page (e.g. `/catalogue/category/books/mystery_3/index.html`)
- Use `ai_extract_from_page` on the raw HTML to get data without CSS selectors — compare accuracy with the selector-based approach
- Add a `robots_allowed(url)` function that checks `/robots.txt` before scraping